# 실습 문제 모범답안 - 103. RAG Agent

`103_RAG_Agent` 노트북 실습 문제에 대한 모범답안 예시입니다.

### 구현 요약

**블로그 QA 에이전트 (인덱싱 + 검색 도구 + RAG 에이전트)**

1. **인덱싱**: `WebBaseLoader`로 블로그 로드 → `RecursiveCharacterTextSplitter`로 분할 → 새 `InMemoryVectorStore`에 저장
2. **검색 도구**: `retrieve_blog_context(query)` - k=3 검색, `content_and_artifact` 형식으로 반환
3. **RAG 에이전트**: 검색된 정보에 근거해서만 답변하도록 system prompt 지정 (환각 방지)
4. **테스트**: 블로그 관련 질문 + 블로그에 없는 질문으로 근거 기반 답변 여부 확인

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 모델 및 임베딩 초기화
model = init_chat_model("gpt-5.4-mini", model_provider="openai")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 인덱싱: 웹 문서 로드 → 분할 → 벡터 스토어 저장

본문의 나이키 벡터 스토어와 섞이지 않도록 **별도의 새 벡터 스토어**를 만듭니다.

In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1) 웹 문서 로드
loader = WebBaseLoader("https://botpress.com/ko/blog/llm-agents")
docs = loader.load()
print(f"로드된 문서 수: {len(docs)}")
print(f"문서 길이: {len(docs[0].page_content)} 문자")

# 2) 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(f"분할된 청크 수: {len(all_splits)}")

# 3) 새 벡터 스토어에 저장 (본문의 나이키 벡터 스토어와 분리)
blog_vector_store = InMemoryVectorStore(embeddings)
document_ids = blog_vector_store.add_documents(documents=all_splits)
print(f"저장된 문서 ID 수: {len(document_ids)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


로드된 문서 수: 1
문서 길이: 9548 문자
분할된 청크 수: 17
저장된 문서 ID 수: 17


## 2. 검색 도구 정의

`response_format="content_and_artifact"` 를 사용하여 LLM에게 전달할 문자열(content)과
원본 문서 객체(artifact)를 함께 반환합니다.

In [4]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_blog_context(query: str):
    """블로그 게시글에서 질문과 관련된 정보를 검색합니다.
    
    Args:
        query: 검색할 질문 또는 키워드
    """
    # 벡터 스토어에서 유사한 문서 검색 (k=3)
    retrieved_docs = blog_vector_store.similarity_search(query, k=3)
    
    # 검색된 문서를 출처 포함 문자열로 직렬화
    serialized = "\n\n".join(
        f"출처: {doc.metadata.get('source', 'N/A')}\n내용: {doc.page_content}"
        for doc in retrieved_docs
    )
    
    return serialized, retrieved_docs

# 도구 단독 테스트
test_result = retrieve_blog_context.invoke("LLM 에이전트의 구성 요소")
print(test_result[:300] + "...")

출처: https://botpress.com/ko/blog/llm-agents
내용: 최신 LLM으로 구동되는 맞춤형 자율 에이전트 구축


지금 시작하기









LLM 에이전트는 어떻게 작동하나요?LLM 에이전트는 LLM의 강력함에 검색, 추론, 메모리, 도구 활용을 결합해 자율적으로 작업을 완수합니다. 각 구성 요소의 역할을 살펴보겠습니다.

검색
대부분의 에이전트는 검색 증강 생성(RAG)을 사용해 실시간 데이터에 접근합니다. 예를 들어, 법률 에이전트는 규제 변경 사항을, 이커머스 에이전트는 상품 재고를 확인할 수 ...


## 3. RAG 에이전트 생성

검색된 정보에 근거해서만 답변하도록 system prompt에 지침을 명시합니다.
이렇게 하면 지식 베이스에 없는 질문에 대해 그럴듯한 답을 지어내는 환각(hallucination)을 줄일 수 있습니다.

In [5]:
from langchain.agents import create_agent

system_prompt = (
    "당신은 블로그 게시글에서 관련 문맥(context)을 검색하는 도구에 접근할 수 있습니다. "
    "사용자의 질문에 답하기 전에 반드시 retrieve_blog_context 도구로 관련 정보를 검색하세요. "
    "반드시 검색된 정보에 근거해서 답변하고, 검색 결과에 없는 내용은 모른다고 답하세요."
)

blog_agent = create_agent(
    model,
    tools=[retrieve_blog_context],
    system_prompt=system_prompt
)

## 4. 테스트

In [7]:
queries = [
    "LLM 에이전트를 정의하는 4가지 특징은 무엇인가요?",
]

for query in queries:
    print(f"\n{'='*80}")
    print(f"질문: {query}")
    print("="*80)
    
    # 스트리밍 방식으로 실행하여 도구 호출 여부를 단계별로 확인
    for event in blog_agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="values",
    ):
        event["messages"][-1].pretty_print()


질문: LLM 에이전트를 정의하는 4가지 특징은 무엇인가요?
================================ Human Message =================================

LLM 에이전트를 정의하는 4가지 특징은 무엇인가요?
================================== Ai Message ==================================
Tool Calls:
  retrieve_blog_context (call_bgef8cuCMwbjmvBPTaXkznf3)
 Call ID: call_bgef8cuCMwbjmvBPTaXkznf3
  Args:
    query: LLM 에이전트를 정의하는 4가지 특징
================================= Tool Message =================================
Name: retrieve_blog_context

출처: https://botpress.com/ko/blog/llm-agents
내용: ‍이러한 기능이 결합되어 LLM 에이전트는 복잡하고 다단계의 워크플로우도 완전히 자율적으로 처리할 수 있습니다.예를 들어:B2B 영업 에이전트는 CRM에서 잠재 고객 데이터를 검색하고, 거래 진행 상황을 분석하며, 과거 상호작용을 기억해 맞춤형 후속 조치를 취하고, 이메일과 캘린더 API를 활용해 발송 및 일정을 잡습니다.IT 에이전트는 시스템 로그를 검색해 오류를 진단하고, 문제 해결 단계를 분석하며, 이전 사용자 이슈에서 효과적이었던 방법을 기억하고, 스크립트를 실행해 서비스를 재시작하거나 티켓을 생성합니다.LLM 에이전트를 정의하는 4가지 특징은 무엇인가요?LLM 에이전트의 핵심 특징 네 가지는 다음과 같습니다.1. 언어 모델언어 모델은 LLM 에이전트의 ‘두뇌’로 여겨집니다. 모델의 품질과 규모가 LLM 에이전트의 성능에 직접적인 영향을 미칩니다.방대한 텍스트 데이터로 학습된 정교한 알고리즘으로, 맥락을 이해하고 